# Spark X2.6 4B Recursive72 — End-to-End Frozen-Backbone `TwoStreamMHC` & 18-Visit Paged `D-LSA` Training Notebook

This 10-cell end-to-end Google Colab workflow trains **all additional recurrent and sparse-attention components** on top of the **100% frozen** `XHToken/Spark-X2.6-4B` (`64ca8f2a295b8ceb2757b9d4749a6558c4759e5b`) 36-layer hybrid backbone and commits **both intermediate stage checkpoints and the final gate-validated release** to **[`TassP0lar/Spark-X2.6-4B-Recursive72`](https://huggingface.co/TassP0lar/Spark-X2.6-4B-Recursive72)**:

1. **Frozen 36L Backbone (`0` trainable base params, full-byte SHA-256 verified)**:
   - 27 Sliding-Window full-RoPE layers (`w=512, theta=1e4`) + 9 Global partial-RoPE (`0.25`, 64 rotated + 192 unrotated dims) layers (`{3, 7, 11, 15, 19, 23, 27, 31, 35}`), native `GELU`, head-wise sigmoid `g_proj` + `out_proj`.
   - Expanded at runtime into the **72-visit Loop72 schedule** (`Prelude L0-3` [4] + `Pass 1 L4-27` [24] + `Pass 2 L4-27` [24] + `Pass 3 L12-23` [12] + `Coda L28-35` [8] = **18 global visits + 54 local visits**).
2. **Trainable Component #1 — 24 Bounded-Gain `TwoStreamMHC` Controllers (`10,078,176` FP32 params, Schema v2)**:
   - Doubly-stochastic $2 \times 2$ Sinkhorn highway mixing (`H_pre, H_post`) + bounded `gain_mode="sigmoid"` (`0 < g_{P2}, g_{P3} < 1`).
   - **Stage A**: Bit-exact state-neutral gradient, activation-checkpoint replay, and 18/18 native Q/K capture gate.
   - **Stage B (`[stage_b_retention]` Hub commit)**: 36L-to-72L teacher-retention KL + assistant-only CE warmup.
   - **Stage C (`[stage_c_dense_mhc]` Hub commit)**: 5-domain multi-pass deliberation training with 3-phase unfreezing and held-out `dev` best-checkpoint selection.
   - **Stage D (`[stage_d_omp_mhc]` Hub commit)**: Pinned Oh My Pi (`omp` v18.18.1, `input` hashline patch) specialization with independent cosine schedule and held-out `dev` non-regression rollback gate.
3. **Trainable Component #2 — 18-Visit `VisitAwareDLSAIndexBank` (`1,769,472` projection params + `36` `log_tau`/`raw_dist_alpha` scalars)**:
   - **Stage E (`[stage_e_selectors]` Hub commit)**: Streamed FP64 Gram PCA initialization + matched-support KL distillation on coherent `8K–16K` documents (`> 4,612` exact cutoff), cryptographically bound to the winning mHC controller SHA-256.
4. **Verification & Final Hub Release (`TassP0lar/Spark-X2.6-4B-Recursive72`)**:
   - **Stage F**: Paged D-LSA inference attachment (`strict_selector_provenance=True`), decode throughput, and physical deduplicated GPU/CPU/NVMe memory ledger.
   - **Clean-Process Export & Final Release Commit**: Isolated subprocess reload verification (with repo checkout stripped from `PYTHONPATH`) followed by the final commit to `TassP0lar/Spark-X2.6-4B-Recursive72`.
   - **Fast Post-Disconnect Recovery (Cell 11)**: If Colab disconnects after Stages B–E have already been pushed to Hub, run **only Cell 1 + Cell 11** to reconstruct, gate-verify, and push the Final Validated Release Commit in ~3–4 minutes without re-training.


In [ ]:
# Cell 1 — Immutable Environment Setup, Hardware Sizing, HF Hub Auth & Full Unit Test Suite
import copy
import importlib
import json
import os
from pathlib import Path
import subprocess
import sys
import time

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_BRANCH = 'loop72-dlsa'
if not Path('train_spark_mhc.py').is_file():
    clone_dir = Path('/content/Spark-X2.6-Recursion')
    if not clone_dir.exists():
        subprocess.run(['git', 'clone', '-b', REPO_BRANCH, REPO_URL, str(clone_dir)], check=True)
    os.chdir(clone_dir)
if Path('.git').exists() and os.environ.get('SPARKX_SKIP_GIT_SYNC') != '1':
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Install runtime/training dependencies if missing (no mutable git pull after Cell 1 setup)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.48.0', 'accelerate', 'datasets', 'safetensors', 'huggingface_hub', 'pyyaml'],
    check=True,
)

# Reload repo modules in dependency order if Cell 1 is re-run inside a live kernel (`sys.modules` cache)
importlib.invalidate_caches()
for _mod_name in ('load_inference_v03', 'train_spark_dlsa', 'train_spark_mhc', 'sparkx_loop72_dlsa_benchmarks'):
    if _mod_name in sys.modules:
        importlib.reload(sys.modules[_mod_name])

import torch
from train_spark_mhc import (
    DEFAULT_HF_MODEL_NAME,
    DEFAULT_HF_REPO_ID,
    MHCTrainingConfig,
    _parameter_sha256,
    apply_hardware_profile_to_config,
    build_completion_gates_report,
    export_and_upload_recursive72_to_hub,
    finalize_release_from_hub_stages,
    normalize_hf_repo_id,
    resolve_hf_token,
    run_mhc_training_pipeline,
    run_stage_a_verification,
    verify_clean_export_directory,
)

RUN_TS = time.strftime('%Y%m%d_%H%M%S', time.gmtime())
RUN_DIR = Path(os.environ.get('SPARKX_RUN_DIR', f'results/loop72_mhc_dlsa_{RUN_TS}')).resolve()
RUN_DIR.mkdir(parents=True, exist_ok=True)

git_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
pip_freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True).strip().splitlines()

HF_TOKEN = resolve_hf_token()
TARGET_REPO_ID = normalize_hf_repo_id(
    os.environ.get('SPARKX_HF_REPO_ID', DEFAULT_HF_REPO_ID),
    token=HF_TOKEN,
)
PUSH_TO_HUB = bool(HF_TOKEN)

# Single-pass hardware sizing: 'auto' detects GPU VRAM (e.g. Colab G4 96GB / A100 / L4) without overwriting explicit overrides
cfg = MHCTrainingConfig(
    output_dir=str(RUN_DIR),
    hardware_profile='auto',
    gain_mode='sigmoid',
    curriculum='fixed_72',
    stage_b_steps=int(os.environ.get('SPARKX_STAGE_B_STEPS', '100')),
    stage_c_steps=int(os.environ.get('SPARKX_STAGE_C_STEPS', '300')),
    stage_d_steps=int(os.environ.get('SPARKX_STAGE_D_STEPS', '60')),
    stage_e_calibration_seqs=int(os.environ.get('SPARKX_STAGE_E_SEQS', '16')),
    stage_e_distill_steps=int(os.environ.get('SPARKX_STAGE_E_STEPS', '300')),
    target_corpus_examples=int(os.environ.get('SPARKX_CORPUS_EXAMPLES', '1200')),
    allow_smoke_fixtures=False,
    hf_model_name=DEFAULT_HF_MODEL_NAME,
    hf_repo_id=TARGET_REPO_ID,
    push_to_hub=PUSH_TO_HUB,
    push_intermediate_stages=True,
    hf_token=HF_TOKEN,
)
HW_PROFILE = apply_hardware_profile_to_config(cfg)

env_manifest = {
    'git_head_sha': git_sha,
    'python_version': sys.version,
    'torch_version': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'hardware_profile': HW_PROFILE,
    'target_hf_repo_id': TARGET_REPO_ID,
    'hf_token_detected': PUSH_TO_HUB,
    'push_intermediate_stages': cfg.push_intermediate_stages,
    'pip_freeze_count': len(pip_freeze),
}
(RUN_DIR / 'environment_manifest.json').write_text(json.dumps(env_manifest, indent=2) + '\n')
(RUN_DIR / 'pip_freeze.txt').write_text('\n'.join(pip_freeze) + '\n')
print('=== Immutable Run Environment & Hardware Sizing ===')
print(json.dumps(env_manifest, indent=2))
if not PUSH_TO_HUB:
    print('WARNING: HF_TOKEN not found in Colab Secrets / env; set HF_TOKEN before training to push commits to', TARGET_REPO_ID)

print('\nRunning full unit test suite before GPU model loading...')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


In [ ]:
# Cell 2 — Real 5-Domain Corpus Construction, Document-Disjoint Split Audit & Deterministic Manifest Freeze
from sparkx_training_data import (
    build_real_five_domain_corpus,
    load_manifest_records,
    save_manifest_records,
    verify_no_split_document_leakage,
)

EXTERNAL_DATA_MANIFEST = os.environ.get('SPARKX_DATA_MANIFEST', '').strip()
MANIFEST_DIR = RUN_DIR / 'data_manifests'
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
PINNED_MANIFEST_PATH = MANIFEST_DIR / 'corpus_all.jsonl'

if EXTERNAL_DATA_MANIFEST:
    loaded_recs, corpus_info = load_manifest_records(EXTERNAL_DATA_MANIFEST)
    splits = {
        'train': [r for r in loaded_recs if r.split == 'train'],
        'dev': [r for r in loaded_recs if r.split == 'dev'],
        'test': [r for r in loaded_recs if r.split == 'test'],
    }
    save_meta = save_manifest_records(loaded_recs, PINNED_MANIFEST_PATH)
    corpus_summary = {'mode': 'external_manifest', 'smoke_fixture_mode': False, **corpus_info, **save_meta}
else:
    splits, corpus_summary = build_real_five_domain_corpus(
        target_total_records=cfg.target_corpus_examples,
        seed=cfg.seed,
        use_hf_streaming=True,
        allow_smoke_fixtures=False,
    )
    all_records = list(splits['train']) + list(splits['dev']) + list(splits['test'])
    save_meta = save_manifest_records(all_records, PINNED_MANIFEST_PATH)
    corpus_summary = {**corpus_summary, **save_meta}

leak_check = verify_no_split_document_leakage(splits['train'], splits['dev'], splits['test'])
assert corpus_summary.get('smoke_fixture_mode') is False, 'Production GPU training forbids smoke fixtures!'
assert leak_check['passed'] is True, f'Split document leakage detected: {leak_check}'
assert len(splits['train']) > 0 and len(splits['dev']) > 0, f'Empty train/dev split: {corpus_summary}'

# Pin all subsequent training stages to this exact SHA-256 JSONL manifest
cfg.data_manifest = str(PINNED_MANIFEST_PATH)
print('=== Real 5-Domain Corpus & Disjoint Document Split Audit ===')
print(json.dumps(corpus_summary, indent=2))


In [ ]:
# Cell 3 — Load Frozen 36L Backbone into 72-Visit Differentiable Runtime + State-Neutral Stage A Verification Gate
from load_inference_v03 import load_sparkx_loop72_mhc_for_training
from sparkx_training_data import DomainWeightedBatchStream, STAGE_C_DOMAIN_WEIGHTS

model, tokenizer = load_sparkx_loop72_mhc_for_training(
    repo_or_dir=cfg.model_id,
    base_repo=cfg.model_id,
    base_revision=cfg.revision,
    token=HF_TOKEN,
)
model.base.requires_grad_(False)

# Full-byte SHA-256 across all 4B frozen Spark backbone parameters (zero head/tail sampling)
BASE_SHA256 = _parameter_sha256(model.base.named_parameters())
assert all(not p.requires_grad for p in model.base.parameters()), 'Backbone parameters must be 100% frozen!'

# Audit exact assistant-only token supervision & padding ledger on the pinned train split
audit_stream = DomainWeightedBatchStream(
    splits['train'], tokenizer, max_seq_len=cfg.seq_len, micro_batch_size=cfg.micro_batch_size,
    domain_weights=STAGE_C_DOMAIN_WEIGHTS, seed=cfg.seed, allow_tail_rewindow=False,
)
_ = audit_stream.next_microbatch(device='cpu')
print('=== Assistant-Only Token Supervision Audit ===')
print(json.dumps(audit_stream.ledger(), indent=2))

# Stage A: State-neutral mechanical, gradient, checkpoint-replay & 18/18 native Q/K capture verification
STAGE_A_REPORT = run_stage_a_verification(
    model,
    cfg=cfg,
    vocab_size=int(model.base.config.vocab_size),
    seq_len=min(64, int(cfg.seq_len)),
    report_path=RUN_DIR / 'stage_a_report.json',
)
print('=== Stage A State-Neutral Verification Report ===')
print(json.dumps(STAGE_A_REPORT, indent=2))
assert STAGE_A_REPORT['passed'] is True, f'Stage A failed: {STAGE_A_REPORT}'
assert STAGE_A_REPORT['state_neutral'] is True and STAGE_A_REPORT['mutated_controller_tensors'] == 0
assert STAGE_A_REPORT['captured_global_visits'] == 18
assert _parameter_sha256(model.base.named_parameters()) == BASE_SHA256


In [ ]:
# Cell 4 — Stage B: 36L-to-72L Teacher Retention Warmup + Intermediate Hub Commit `[stage_b_retention]`
cfg_b = copy.deepcopy(cfg)
REPORT_B = run_mhc_training_pipeline(
    model,
    tokenizer,
    cfg=cfg_b,
    run_stage_a=False,
    run_stage_b=True,
    run_stage_c=False,
    run_stage_d=False,
    run_stage_e=False,
    reset_controllers=True,
    run_final_export=False,
)
assert _parameter_sha256(model.base.named_parameters()) == BASE_SHA256, 'Frozen backbone mutated during Stage B!'
print('=== Stage B Teacher Retention Summary & Intermediate Hub Commit ===')
print(json.dumps({
    'stage_b': {k: v for k, v in REPORT_B['stages']['stage_b'].items() if k != 'eval_history'},
    'hub_commit': REPORT_B['intermediate_hub_commits'][-1] if REPORT_B['intermediate_hub_commits'] else None,
    'frozen_base_sha256_verified': True,
}, indent=2, default=str))


In [ ]:
# Cell 5 — Stage C: Multi-Pass Deliberation Training (Staged Unfreezing + Best Dev Selection) + Hub Commit `[stage_c_dense_mhc]`
cfg_c = copy.deepcopy(cfg)
cfg_c.resume_from_checkpoint = str(RUN_DIR / 'stage_b_retention_mhc')
REPORT_C = run_mhc_training_pipeline(
    model,
    tokenizer,
    cfg=cfg_c,
    run_stage_a=False,
    run_stage_b=False,
    run_stage_c=True,
    run_stage_d=False,
    run_stage_e=False,
    reset_controllers=False,
    run_final_export=False,
)
assert _parameter_sha256(model.base.named_parameters()) == BASE_SHA256, 'Frozen backbone mutated during Stage C!'
print('=== Stage C Deliberation Training Summary & Intermediate Hub Commit ===')
print(json.dumps({
    'stage_c': {k: v for k, v in REPORT_C['stages']['stage_c'].items() if k != 'eval_history'},
    'hub_commit': REPORT_C['intermediate_hub_commits'][-1] if REPORT_C['intermediate_hub_commits'] else None,
    'frozen_base_sha256_verified': True,
}, indent=2, default=str))


In [ ]:
# Cell 6 — Stage D: Pinned OMP v18.18.1 Tool Specialization + Dev Non-Regression Rollback Gate + Hub Commit `[stage_d_omp_mhc]`
cfg_d = copy.deepcopy(cfg)
cfg_d.resume_from_checkpoint = str(RUN_DIR / 'stage_c_dense_mhc')
REPORT_D = run_mhc_training_pipeline(
    model,
    tokenizer,
    cfg=cfg_d,
    run_stage_a=False,
    run_stage_b=False,
    run_stage_c=False,
    run_stage_d=True,
    run_stage_e=False,
    reset_controllers=False,
    run_final_export=False,
)
assert _parameter_sha256(model.base.named_parameters()) == BASE_SHA256, 'Frozen backbone mutated during Stage D!'
SELECTED_MHC_META = json.loads((RUN_DIR / 'selected_mhc_checkpoint.json').read_text())
print('=== Stage D OMP v18.18.1 Summary, Dev Rollback Gate & Intermediate Hub Commit ===')
print(json.dumps({
    'stage_d': {k: v for k, v in REPORT_D['stages']['stage_d'].items() if k != 'eval_history'},
    'selected_mhc_checkpoint': SELECTED_MHC_META,
    'hub_commit': REPORT_D['intermediate_hub_commits'][-1] if REPORT_D['intermediate_hub_commits'] else None,
    'frozen_base_sha256_verified': True,
}, indent=2, default=str))


In [ ]:
# Cell 7 — Stage E: 18/18 Global-Visit Streamed-Gram PCA + Matched-Support KL Distillation + Hub Commit `[stage_e_selectors]`
cfg_e = copy.deepcopy(cfg)
cfg_e.resume_from_checkpoint = str(SELECTED_MHC_META['checkpoint_dir'])
REPORT_E = run_mhc_training_pipeline(
    model,
    tokenizer,
    cfg=cfg_e,
    run_stage_a=False,
    run_stage_b=False,
    run_stage_c=False,
    run_stage_d=False,
    run_stage_e=True,
    reset_controllers=False,
    run_final_export=False,
)
st_e = REPORT_E['stages']['stage_e']
assert _parameter_sha256(model.base.named_parameters()) == BASE_SHA256, 'Frozen backbone mutated during Stage E!'
assert st_e['trained_global_visits'] == 18 and st_e['skipped_global_visits'] == 0, f'Incomplete Stage E visits: {st_e}'
assert st_e['calibration_data']['exceeds_exact_4612_cutoff'] is True, f'Stage E calibration must exceed 4612 tokens: {st_e}'
assert st_e['mean_final_loss'] <= st_e['mean_initial_loss'], f'Stage E selector KL did not improve: {st_e}'
print('=== Stage E 18-Visit D-LSA Distillation Summary & Intermediate Hub Commit ===')
print(json.dumps({
    'trained_global_visits': st_e['trained_global_visits'],
    'skipped_global_visits': st_e['skipped_global_visits'],
    'mean_initial_loss': st_e['mean_initial_loss'],
    'mean_final_loss': st_e['mean_final_loss'],
    'mean_mass_recall_at_2048': st_e.get('mean_mass_recall_at_2048'),
    'mean_bucket_runtime_recall': st_e['mean_bucket_runtime_recall'],
    'provenance': st_e['provenance'],
    'hub_commit': REPORT_E['intermediate_hub_commits'][-1] if REPORT_E['intermediate_hub_commits'] else None,
}, indent=2, default=str))


In [ ]:
# Cell 8 — Stage F: Paged D-LSA Sparse Runtime Attachment, Provenance Lock, Decode Speed & Physical Memory Ledger
from load_inference_v03 import configure_model_for_loop72_lsa
import sparkx_loop72_dlsa_benchmarks as bench

SELECTOR_CKPT_PATH = RUN_DIR / 'visit_selectors_stage_e.pt'
configure_model_for_loop72_lsa(
    model,
    dlsa_checkpoint=str(SELECTOR_CKPT_PATH),
    recurrence_mode='learned_mhc',
    strict_selector_provenance=True,
    require_verified_selectors=True,
)
EXEC_MANIFEST = model.execution_manifest()
assert EXEC_MANIFEST['selector_checkpoint_status']['initialization'] == 'checkpoint_loaded_verified'
assert EXEC_MANIFEST['selector_checkpoint_status']['trained_global_visits'] == 18

SPEED_REPORT = bench.benchmark_loop72_decode_speed(
    model, tokenizer, prompt_lengths=(512, 2048, 8192),
    gen_tokens=128, warmups=1, repeats=2, verbose=True,
)
DIAG_SMOKE = bench.evaluate_selected_benchmark_tasks(
    model, tokenizer, max_new_tokens=1024,
    include_hf_streamed=False, include_smoke=True, task_mode='diagnostic', verbose=True,
)
STAGE_F_SUMMARY = {
    'execution_manifest': EXEC_MANIFEST,
    'speed_horizons': SPEED_REPORT.get('horizons'),
    'diagnostic_smoke_by_benchmark': DIAG_SMOKE.get('by_benchmark'),
}
(RUN_DIR / 'stage_f_eval.json').write_text(json.dumps(STAGE_F_SUMMARY, indent=2, default=str) + '\n')
print('=== Stage F Paged D-LSA Runtime & Physical Memory Ledger Summary ===')
print(json.dumps(STAGE_F_SUMMARY['speed_horizons'], indent=2, default=str))


In [ ]:
# Cell 9 — Clean-Process Standalone Export Packaging & Isolated Subprocess Reload Parity Gate
import gc
from load_inference_v03 import load_sparkx_loop72_lsa

EXPORT_DIR = RUN_DIR / 'final_loop72_mhc_dlsa_export'
cfg_local_pkg = copy.deepcopy(cfg_e)
cfg_local_pkg.push_to_hub = False  # Package & gate-verify locally first before final Hub release commit

LOCAL_PKG_REPORT = export_and_upload_recursive72_to_hub(
    model,
    tokenizer,
    export_dir=EXPORT_DIR,
    cfg=cfg_local_pkg,
    report=REPORT_E,
)
CLEAN_SUBPROCESS_CHECK = verify_clean_export_directory(EXPORT_DIR)
print('=== Isolated Clean-Subprocess Export Verification (Repo Stripped from PYTHONPATH) ===')
print(json.dumps(CLEAN_SUBPROCESS_CHECK, indent=2))
assert CLEAN_SUBPROCESS_CHECK['passed'] is True and CLEAN_SUBPROCESS_CHECK['bad_imports'] == []

# Verify full in-process reload from standalone export directory and greedy next-token parity
probe_ids = tokenizer('Verify deterministic Loop72 + D-LSA reload parity: 2 + 2 =', return_tensors='pt')['input_ids'].to(
    next(model.base.parameters()).device
)
with torch.no_grad():
    orig_out = model.generate_cached(
        probe_ids, max_new_tokens=8, do_sample=False, repetition_penalty=1.0,
        force_think_close=False, stream_print=False, eos_token_id=tokenizer.eos_token_id,
    )

reloaded_model, _ = load_sparkx_loop72_lsa(
    repo_or_dir=str(EXPORT_DIR),
    recurrence_mode='learned_mhc',
    strict_selector_provenance=True,
    require_verified_selectors=True,
)
assert _parameter_sha256(reloaded_model.base.named_parameters()) == BASE_SHA256
rel_sel_status = reloaded_model.execution_manifest()['selector_checkpoint_status']
assert rel_sel_status['initialization'] in (
    'checkpoint_loaded_verified',
    'loop_state_checkpoint_loaded_verified',
), f'Unexpected selector initialization status: {rel_sel_status}'
assert rel_sel_status.get('verified') is True
with torch.no_grad():
    rel_out = reloaded_model.generate_cached(
        probe_ids, max_new_tokens=8, do_sample=False, repetition_penalty=1.0,
        force_think_close=False, stream_print=False, eos_token_id=tokenizer.eos_token_id,
    )
assert torch.equal(orig_out.cpu(), rel_out.cpu()), f'Reload greedy token mismatch: {orig_out.tolist()} vs {rel_out.tolist()}'

del reloaded_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Exact standalone reload greedy token parity & frozen base SHA-256 verified!')


In [ ]:
# Cell 10 — Final Gate Audit & Validated Release Commit to `TassP0lar/Spark-X2.6-4B-Recursive72`
COMPLETION_GATES = build_completion_gates_report(REPORT_E, clean_reload=CLEAN_SUBPROCESS_CHECK)
print('=== Completion Gates Audit ===')
print(json.dumps(COMPLETION_GATES, indent=2))

assert COMPLETION_GATES['frozen_backbone_verified'] is True
assert COMPLETION_GATES['stage_a_state_neutral_passed'] is True
assert COMPLETION_GATES['stage_e_all_18_visits_calibrated'] is True
assert COMPLETION_GATES['clean_export_subprocess_verified'] is True
assert _parameter_sha256(model.base.named_parameters()) == BASE_SHA256

FINAL_HUB_RELEASE = export_and_upload_recursive72_to_hub(
    model,
    tokenizer,
    export_dir=EXPORT_DIR,
    cfg=cfg,
    report=REPORT_E,
)
REPORT_E['stage_f'] = STAGE_F_SUMMARY
REPORT_E['completion_gates'] = COMPLETION_GATES
REPORT_E['huggingface_export'] = FINAL_HUB_RELEASE
(RUN_DIR / 'mhc_training_report.json').write_text(json.dumps(REPORT_E, indent=2, default=str) + '\n')

print('=== Final Validated Hub Release Summary ===')
print(json.dumps({
    'hf_repo_id': FINAL_HUB_RELEASE['hf_repo_id'],
    'uploaded_final_commit': FINAL_HUB_RELEASE['uploaded'],
    'intermediate_commits_pushed': len([c for c in REPORT_E['intermediate_hub_commits'] if c.get('uploaded')]),
    'base_sha256_immutable': BASE_SHA256,
    'export_dir': str(EXPORT_DIR),
}, indent=2, default=str))


In [ ]:
# Cell 11 — Self-Contained Post-Disconnect Recovery & Final Hub Release (Works Standalone OR After Cell 1!)
# Syncs latest `origin/loop72-dlsa`, reloads cached `sys.modules` in-place, pulls saved Stage B/C/D/E manifests &
# weights from `TassP0lar/Spark-X2.6-4B-Recursive72`, runs Stage A + Stage F + clean-subprocess + greedy parity gates,
# and pushes the Final Validated Release Commit to Hugging Face Hub in ~3–4 minutes (zero re-training!).
import gc, importlib, json, os, subprocess, sys, time
from pathlib import Path
import torch

if not Path('train_spark_mhc.py').is_file():
    _cdir = Path('/content/Spark-X2.6-Recursion')
    if not _cdir.exists():
        subprocess.run(['git', 'clone', '-b', 'loop72-dlsa', 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git', str(_cdir)], check=True)
    os.chdir(_cdir)
if Path('.git').exists() and os.environ.get('SPARKX_SKIP_GIT_SYNC') != '1':
    subprocess.run(['git', 'fetch', 'origin', 'loop72-dlsa'], check=True)
    subprocess.run(['git', 'reset', '--hard', 'origin/loop72-dlsa'], check=True)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

importlib.invalidate_caches()
for _m in ('load_inference_v03', 'train_spark_dlsa', 'train_spark_mhc', 'sparkx_loop72_dlsa_benchmarks'):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from train_spark_mhc import (
    DEFAULT_HF_MODEL_NAME,
    DEFAULT_HF_REPO_ID,
    MHCTrainingConfig,
    apply_hardware_profile_to_config,
    finalize_release_from_hub_stages,
    normalize_hf_repo_id,
    resolve_hf_token,
)

# Free any lingering duplicate `reloaded_model` from an interrupted Cell 9 run
if 'reloaded_model' in globals():
    del reloaded_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

if 'cfg' not in globals():
    _tok_hf = resolve_hf_token()
    _run_dir = Path(os.environ.get('SPARKX_RUN_DIR', f"results/loop72_finalize_{time.strftime('%Y%m%d_%H%M%S', time.gmtime())}")).resolve()
    cfg = MHCTrainingConfig(
        output_dir=str(_run_dir),
        hardware_profile='auto',
        gain_mode='sigmoid',
        curriculum='fixed_72',
        allow_smoke_fixtures=False,
        hf_model_name=DEFAULT_HF_MODEL_NAME,
        hf_repo_id=normalize_hf_repo_id(os.environ.get('SPARKX_HF_REPO_ID', DEFAULT_HF_REPO_ID), token=_tok_hf),
        push_to_hub=bool(_tok_hf),
        hf_token=_tok_hf,
    )
    apply_hardware_profile_to_config(cfg)

FINALIZED_REPORT = finalize_release_from_hub_stages(
    cfg=cfg,
    model=globals().get('model'),
    tokenizer=globals().get('tokenizer'),
    run_stage_a=True,
    run_stage_f=True,
    verify_local_reload_parity=True,
)
print('=== Post-Disconnect Finalized Completion Gates ===')
print(json.dumps(FINALIZED_REPORT['completion_gates'], indent=2))
assert FINALIZED_REPORT['completion_gates']['production_release_ready'] is True
print('=== Final Validated Hub Release Summary ===')
print(json.dumps({
    'hf_repo_id': FINALIZED_REPORT['huggingface_export']['hf_repo_id'],
    'uploaded_final_commit': FINALIZED_REPORT['huggingface_export']['uploaded'],
    'selected_mhc_stage': FINALIZED_REPORT['selected_mhc_stage'],
    'base_sha256_immutable': FINALIZED_REPORT['base_sha256_post_mhc'],
    'export_dir': FINALIZED_REPORT['huggingface_export']['export_dir'],
}, indent=2, default=str))
